In [1]:
!pip show kfp

Name: kfp
Version: 2.9.0
Summary: Kubeflow Pipelines SDK
Home-page: https://github.com/kubeflow/pipelines
Author: The Kubeflow Authors
Author-email: 
License: 
Location: /home/amit.wankhede@GSLAB.COM/Work/PoC/GenAI/ai-examples/k8s-diagnostic-agent/rag-example/example2/.venv/lib/python3.10/site-packages
Requires: click, docstring-parser, google-api-core, google-auth, google-cloud-storage, kfp-pipeline-spec, kfp-server-api, kubernetes, protobuf, PyYAML, requests-toolbelt, tabulate, urllib3
Required-by: kfp-kubernetes


In [2]:

import kfp
from kfp import dsl
from kfp.dsl import component, Input, Output, Artifact
from kfp.kubernetes import mount_pvc

In [3]:
@component(
    base_image='amitinfo2k/pcap-pipeline:3.10',
    packages_to_install=['scapy']
)
def parse_tcpdump(
    input_pcap: str,
    output_text: Output[Artifact]
):
    from scapy.all import rdpcap
    
    # Read the pcap file
    packets = rdpcap(input_pcap)
    
    # Generate text summaries for each packet
    summaries = [pkt.summary() for pkt in packets]
    
    # Write summaries to output artifact
    with open(output_text.path, 'w') as f:
        f.write('\n'.join(summaries))

In [4]:
@component(
    base_image='amitinfo2k/pcap-pipeline:3.10',
)
def embed_and_store(
    input_text: Input[Artifact],
    collection_name: str
):
    import chromadb
    from sentence_transformers import SentenceTransformer
    
    # Connect to ChromaDB service with v2 API and explicit tenant/database
    client = chromadb.HttpClient(
        host='chroma-service.chromedb.svc.cluster.local',
        port=8000,
        ssl=False,
        headers={
            "x-chroma-client-version": "2.0.0",
            "x-chroma-request-source": "kubeflow-pipeline"
        },
        tenant="default_tenant",
        database="default_database"
    )
    
    # Get or create collection with v2 API
    try:
        collection = client.get_collection(name=collection_name)
    except ValueError:
        # Collection doesn't exist, create it
        collection = client.create_collection(name=collection_name)
    
    # Load embedding model
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Read packet summaries
    with open(input_text.path, 'r') as f:
        texts = [line.strip() for line in f.readlines() if line.strip()]
    
    # Generate embeddings
    embeddings = model.encode(texts).tolist()
    
    # Add to ChromaDB
    collection.add(
        documents=texts,
        embeddings=embeddings,
        ids=[f'pkt_{i}' for i in range(len(texts))]
    )

In [7]:
@component(
    base_image='amitinfo2k/pcap-pipeline:3.10',
    packages_to_install=['google-generativeai']
)
def rag_analysis(
    collection_name: str,
    query: str,
    output_analysis: Output[Artifact]
):
    import chromadb
    from sentence_transformers import SentenceTransformer
    import google.generativeai as genai
    import os
    
    # Configure Gemini API key (assume it's set as env var or secret)
    genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
    
    # Connect to ChromaDB service with v2 API and explicit tenant/database
    client = chromadb.HttpClient(
        host='chroma-service.chromedb.svc.cluster.local',
        port=8000,
        ssl=False,
        headers={
            "x-chroma-client-version": "2.0.0",
            "x-chroma-request-source": "kubeflow-pipeline"
        },
        tenant="default_tenant",
        database="default_database"
    )

    # Get collection with v2 API
    try:
        collection = client.get_collection(name=collection_name)
    except ValueError:
        raise ValueError(f"Collection '{collection_name}' does not exist")
    
    # Load embedding model
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Embed query
    query_embedding = model.encode(query).tolist()
    
    # Retrieve top relevant documents
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )
    
    # Build context from retrieved documents
    context = '\n'.join(results['documents'][0])
    
    # RAG prompt
    prompt = f"Analyze the network traffic based on this context:\n{context}\n\nQuery: {query}\nAnalysis:"
    
    # Generate response using Gemini (placeholder model; adjust as needed, e.g., 'gemini-1.5-flash')
    gemini_model = genai.GenerativeModel('gemini-1.5-flash')
    response = gemini_model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            max_output_tokens=300,
            temperature=0.7
        )
    )
    
    # Write analysis to output
    with open(output_analysis.path, 'w') as f:
        f.write(response.text.strip())

In [8]:
#Define the Kubeflow Pipeline
@dsl.pipeline(
    name='RAG Pipeline for TCPDump Analysis',
    description='A pipeline that processes tcpdump data, stores embeddings in ChromaDB, and performs RAG-based analysis.'
)
def rag_tcpdump_pipeline(
    input_pcap: str = '/mnt/pcap/sample.pcap',  # Path in mounted volume
    collection_name: str = 'tcpdump_collection',
    analysis_query: str = 'Detect any anomalies in the network traffic',
    pvc_name: str = 'pcap-storage'  # Name of the PVC
):
    # Step 1: Parse tcpdump to text summaries
    parse_step = parse_tcpdump(input_pcap=input_pcap)
    
    # Mount the PVC to the step and set image pull policy
    from kfp.kubernetes import mount_pvc, set_image_pull_policy
    mount_pvc(
        task=parse_step,
        pvc_name=pvc_name,
        mount_path='/mnt/pcap'
    )
    set_image_pull_policy(parse_step, 'IfNotPresent')
  # Step 2: Embed and store in ChromaDB
    embed_step = embed_and_store(
        input_text=parse_step.output,
        collection_name=collection_name
    )
    set_image_pull_policy(embed_step, 'IfNotPresent')
    
    # Step 3: Perform RAG analysis
    analysis_step = rag_analysis(
        collection_name=collection_name,
        query=analysis_query
    ).after(embed_step)
    set_image_pull_policy(analysis_step, 'IfNotPresent')

In [ ]:
import re
from urllib.parse import urlsplit, urlencode

import kfp
import requests
import urllib3


class KFPClientManager:
    """
    A class that creates `kfp.Client` instances with Dex authentication.
    """

    def __init__(
        self,
        api_url: str,
        dex_username: str,
        dex_password: str,
        dex_auth_type: str = "local",
        skip_tls_verify: bool = False,
    ):
        """
        Initialize the KfpClient

        :param api_url: the Kubeflow Pipelines API URL
        :param skip_tls_verify: if True, skip TLS verification
        :param dex_username: the Dex username
        :param dex_password: the Dex password
        :param dex_auth_type: the auth type to use if Dex has multiple enabled, one of: ['ldap', 'local']
        """
        self._api_url = api_url
        self._skip_tls_verify = skip_tls_verify
        self._dex_username = dex_username
        self._dex_password = dex_password
        self._dex_auth_type = dex_auth_type
        self._client = None

        # disable SSL verification, if requested
        if self._skip_tls_verify:
            urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

        # ensure `dex_default_auth_type` is valid
        if self._dex_auth_type not in ["ldap", "local"]:
            raise ValueError(
                f"Invalid `dex_auth_type` '{self._dex_auth_type}', must be one of: ['ldap', 'local']"
            )

    def _get_session_cookies(self) -> str:
        """
        Get the session cookies by authenticating against Dex
        :return: a string of session cookies in the form "key1=value1; key2=value2"
        """

        # use a persistent session (for cookies)
        s = requests.Session()

        # GET the api_url, which should redirect to Dex
        resp = s.get(
            self._api_url, allow_redirects=True, verify=not self._skip_tls_verify
        )
        if resp.status_code == 200:
            pass
        elif resp.status_code == 403:
            # if we get 403, we might be at the oauth2-proxy sign-in page
            # the default path to start the sign-in flow is `/oauth2/start?rd=<url>`
            url_obj = urlsplit(resp.url)
            url_obj = url_obj._replace(
                path="/oauth2/start", query=urlencode({"rd": url_obj.path})
            )
            resp = s.get(
                url_obj.geturl(), allow_redirects=True, verify=not self._skip_tls_verify
            )
        else:
            raise RuntimeError(
                f"HTTP status code '{resp.status_code}' for GET against: {self._api_url}"
            )

        # if we were NOT redirected, then the endpoint is unsecured
        if len(resp.history) == 0:
            # no cookies are needed
            return ""

        # if we are at `../auth` path, we need to select an auth type
        url_obj = urlsplit(resp.url)
        if re.search(r"/auth$", url_obj.path):
            url_obj = url_obj._replace(
                path=re.sub(r"/auth$", f"/auth/{self._dex_auth_type}", url_obj.path)
            )

        # if we are at `../auth/xxxx/login` path, then we are at the login page
        if re.search(r"/auth/.*/login$", url_obj.path):
            dex_login_url = url_obj.geturl()
        else:
            # otherwise, we need to follow a redirect to the login page
            resp = s.get(
                url_obj.geturl(), allow_redirects=True, verify=not self._skip_tls_verify
            )
            if resp.status_code != 200:
                raise RuntimeError(
                    f"HTTP status code '{resp.status_code}' for GET against: {url_obj.geturl()}"
                )
            dex_login_url = resp.url

        # attempt Dex login
        resp = s.post(
            dex_login_url,
            data={"login": self._dex_username, "password": self._dex_password},
            allow_redirects=True,
            verify=not self._skip_tls_verify,
        )
        if resp.status_code != 200:
            raise RuntimeError(
                f"HTTP status code '{resp.status_code}' for POST against: {dex_login_url}"
            )

        # if we were NOT redirected, then the login credentials were probably invalid
        if len(resp.history) == 0:
            raise RuntimeError(
                f"Login credentials are probably invalid - "
                f"No redirect after POST to: {dex_login_url}"
            )

        # if we are at `../approval` path, we need to approve the login
        url_obj = urlsplit(resp.url)
        if re.search(r"/approval$", url_obj.path):
            dex_approval_url = url_obj.geturl()

            # approve the login
            resp = s.post(
                dex_approval_url,
                data={"approval": "approve"},
                allow_redirects=True,
                verify=not self._skip_tls_verify,
            )
            if resp.status_code != 200:
                raise RuntimeError(
                    f"HTTP status code '{resp.status_code}' for POST against: {url_obj.geturl()}"
                )

        return "; ".join([f"{c.name}={c.value}" for c in s.cookies])

    def _create_kfp_client(self) -> kfp.Client:
        try:
            session_cookies = self._get_session_cookies()
        except Exception as ex:
            raise RuntimeError(f"Failed to get Dex session cookies") from ex

        # monkey patch the kfp.Client to support disabling SSL verification
        # kfp only added support in v2: https://github.com/kubeflow/pipelines/pull/7174
        original_load_config = kfp.Client._load_config

        def patched_load_config(client_self, *args, **kwargs):
            config = original_load_config(client_self, *args, **kwargs)
            config.verify_ssl = not self._skip_tls_verify
            return config

        patched_kfp_client = kfp.Client
        patched_kfp_client._load_config = patched_load_config

        return patched_kfp_client(
            host=self._api_url,
            cookies=session_cookies,
        )

    def create_kfp_client(self) -> kfp.Client:
        """Get a newly authenticated Kubeflow Pipelines client."""
        return self._create_kfp_client()

# initialize a KFPClientManager
kfp_client_manager = KFPClientManager(
    api_url="http://localhost:8080/pipeline",
    skip_tls_verify=True,

    dex_username="user@example.com",
    dex_password="12341234",

    # can be 'ldap' or 'local' depending on your Dex configuration
    dex_auth_type="local",
)

# get a newly authenticated KFP client
# TIP: long-lived sessions might need to get a new client when their session expires
client = kfp_client_manager.create_kfp_client()

# test the client by listing experiments
experiments = client.list_experiments(namespace="kubeflow-user-example-com")
print(experiments)

#client = kfp.Client(namespace="kubeflow")